# 01 — Data Quality and Cleaning

This notebook performs the data-curation stage of the COPD exacerbation forecasting pipeline.

The main objectives are to:

- load the aggregated daily dataset;
- restrict the analysis to the study period (2010–2023);
- verify the integrity of the daily time series;
- validate missing values generated by temporal lag boundaries;
- identify physically implausible atmospheric pressure measurements;
- analyse the distribution and temporal structure of missing values;
- explore whether missingness is associated with the response variable;
- remove observations affected by the maximum lag boundary;
- impute remaining missing environmental values using linear interpolation.

The resulting cleaned dataset is used in the subsequent exploratory analysis and machine-learning stages.

In [1]:
import pandas as pd
from scipy.stats import spearmanr, pearsonr, fisher_exact

from copd_forecasting.config import (
    RAW_DATA_FILE,
    PROCESSED_DATA_FILE,
    DATE_COLUMN,
    STUDY_START,
    STUDY_END,
    MAX_LAG,
)

from copd_forecasting.data import (
    load_daily_data,
    filter_study_period,
    validate_daily_timeline,
    validate_lag_boundary_missingness,
    replace_invalid_pressure_values,
    remove_lag_boundary,
    impute_lagged_features,
    check_remaining_missing_values,
)

## 1. Data loading

The original dataset contains several worksheets at different levels of aggregation.  
Only the aggregated daily COPD dataset is required for this forecasting pipeline.

Patient-level and individual attendance-level records are therefore not loaded.

In [2]:
df = load_daily_data(RAW_DATA_FILE)

print(f"Raw dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(
    f"Available date range: "
    f"{df[DATE_COLUMN].min().date()} → "
    f"{df[DATE_COLUMN].max().date()}"
)

Raw dataset shape: 5,478 rows × 187 columns
Available date range: 2010-01-01 → 2024-12-30


## 2. Study period

The analysis is restricted to the period defined for the study, from 1 January 2010 to 31 December 2023.

Observations outside this interval are excluded before further data-quality checks.

In [3]:
df = filter_study_period(df)

print(f"Study dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(
    f"Study period: "
    f"{df[DATE_COLUMN].min().date()} → "
    f"{df[DATE_COLUMN].max().date()}"
)

Study dataset shape: 5,113 rows × 187 columns
Study period: 2010-01-01 → 2023-12-31


## 3. Temporal integrity checks

Before analysing missing values, the temporal structure of the dataset is verified.

First, the daily time series is checked for duplicated or missing dates. Second, missing values in lagged environmental variables are evaluated to confirm that the additional missing observations are exclusively caused by the expected temporal boundary effect.

For a variable with lag \(k\), the expected relationship is:

\[
n_{\text{missing}}(LAG_k) =
n_{\text{missing}}(LAG_0) + k
\]

This distinction is important because boundary missingness is a structural consequence of lag construction rather than a true data-quality problem.

In [4]:
validate_daily_timeline(df)

print("✓ Daily timeline is complete: no duplicated or missing dates.")

✓ Daily timeline is complete: no duplicated or missing dates.


In [5]:
lag_validation = validate_lag_boundary_missingness(df)

n_checks = len(lag_validation)
n_failed = (~lag_validation["valid"]).sum()

print(f"Total lag checks: {n_checks}")
print(f"Failed checks: {n_failed}")

if n_failed == 0:
    print(
        "✓ All additional missing values in lagged variables "
        "are consistent with the expected boundary effect."
    )
else:
    display(
        lag_validation.loc[~lag_validation["valid"]]
    )

Total lag checks: 147
Failed checks: 0
✓ All additional missing values in lagged variables are consistent with the expected boundary effect.


## 4. Physically implausible pressure values

Atmospheric pressure measurements were reviewed for physical plausibility.

Values below **800 hPa** were considered incompatible with surface atmospheric pressure measurements in the geographical context of the study. These observations are treated as missing values and subsequently handled together with the remaining missing environmental data.

Only the anomalous pressure measurements are replaced; all other variables recorded on the same day are preserved.

In [6]:
pressure_lag0_cols = [
    col
    for col in [
        "pres_min_LAG0",
        "pres_med_LAG0",
        "pres_max_LAG0",
    ]
    if col in df.columns
]

invalid_pressure_counts = (
    (df[pressure_lag0_cols] < 800)
    .sum()
    .rename("Invalid values")
)

invalid_pressure_counts

pres_min_LAG0    1
pres_med_LAG0    1
pres_max_LAG0    1
Name: Invalid values, dtype: int64

In [7]:
df = replace_invalid_pressure_values(df)

print("✓ Physically implausible pressure values replaced with NaN.")

✓ Physically implausible pressure values replaced with NaN.


## 5. Missing-data overview

Missingness is analysed using the environmental variables corresponding to the day of exposure (`LAG0`).

Only `LAG0` variables are used for this descriptive analysis because the remaining lagged variables contain the same environmental series shifted in time, together with the structural boundary missingness evaluated above.

In [8]:
lag0_cols = [
    col
    for col in df.columns
    if col.endswith("_LAG0")
]

missing_summary = (
    df[lag0_cols]
    .isna()
    .sum()
    .to_frame("N_missing")
)

missing_summary["Percentage"] = (
    missing_summary["N_missing"]
    / len(df)
    * 100
)

missing_summary = (
    missing_summary
    .sort_values(
        "Percentage",
        ascending=False,
    )
)

missing_summary.loc[
    missing_summary["N_missing"] > 0
].round(3)

,N_missing,Percentage
CO_LAG0,136,2.660
PM2p5_LAG0,48,0.939
O3_LAG0,16,0.313
PM10_LAG0,15,0.293
NO_LAG0,9,0.176
NO2_LAG0,9,0.176
hume_max_LAG0,5,0.098
pres_med_LAG0,3,0.059
pres_min_LAG0,3,0.059
pres_max_LAG0,3,0.059


## 6. Association between missingness and the response

To explore whether missing environmental observations are related to the response, a binary missingness indicator is created for each `LAG0` environmental variable.

Three complementary analyses are performed:

- **Spearman correlation** between missingness and the daily number of COPD-related emergency visits;
- **Phi coefficient** between missingness and the binary response;
- **Fisher's exact test** on the corresponding 2 × 2 contingency table.

These analyses are exploratory and do not establish the missing-data mechanism by themselves.

In [9]:
target_binary = (df["n_urgencias"] >= 1).astype(int)

missingness_results = []

for col in lag0_cols:
    if not df[col].isna().any():
        continue

    missing_indicator = df[col].isna().astype(int)

    # Association with daily ED visit count
    rho_count, p_count = spearmanr(
        missing_indicator,
        df["n_urgencias"],
        nan_policy="omit",
    )

    # Phi coefficient:
    # Pearson correlation between two binary variables
    phi_binary, p_phi = pearsonr(
        missing_indicator,
        target_binary,
    )

    # 2 x 2 contingency table
    contingency_table = pd.crosstab(
        missing_indicator,
        target_binary,
    ).reindex(
        index=[0, 1],
        columns=[0, 1],
        fill_value=0,
    )

    odds_ratio, p_fisher = fisher_exact(
        contingency_table.to_numpy()
    )

    missingness_results.append(
        {
            "Variable": col.replace("_LAG0", ""),
            "N_missing": int(missing_indicator.sum()),
            "Percentage": missing_indicator.mean() * 100,
            "rho_count": rho_count,
            "p_count": p_count,
            "phi_binary": phi_binary,
            "p_phi": p_phi,
            "odds_ratio": odds_ratio,
            "p_fisher": p_fisher,
        }
    )

missingness_results = (
    pd.DataFrame(missingness_results)
    .sort_values("Percentage", ascending=False)
    .reset_index(drop=True)
)

missingness_results.round(4)

,Variable,N_missing,Percentage,rho_count,p_count,phi_binary,p_phi,odds_ratio,p_fisher
0,CO,136,2.6599,0.0142,0.3101,0.0181,0.1969,1.2570,0.2215
1,PM2p5,48,0.9388,-0.0091,0.5159,-0.0136,0.3315,0.7550,0.3825
2,O3,16,0.3129,0.0301,0.0313,0.0367,0.0086,5.7836,0.0099
3,PM10,15,0.2934,0.0088,0.5307,0.0129,0.3576,1.6479,0.4412
4,NO2,9,0.1760,0.0197,0.1592,0.0287,0.0401,6.5985,0.0469
5,NO,9,0.1760,0.0197,0.1592,0.0287,0.0401,6.5985,0.0469
6,hume_max,5,0.0978,0.0120,0.3893,0.0032,0.8173,1.2345,1.0000
7,pres_min,3,0.0587,0.0059,0.6715,0.0057,0.6811,1.6461,1.0000
8,pres_max,3,0.0587,0.0059,0.6715,0.0057,0.6811,1.6461,1.0000
9,pres_med,3,0.0587,0.0059,0.6715,0.0057,0.6811,1.6461,1.0000


In [10]:
significant_missingness = missingness_results.loc[
    (missingness_results["p_count"] < 0.05)
    | (missingness_results["p_phi"] < 0.05)
    | (missingness_results["p_fisher"] < 0.05)
]

significant_missingness.round(4)

,Variable,N_missing,Percentage,rho_count,p_count,phi_binary,p_phi,odds_ratio,p_fisher
2,O3,16,0.3129,0.0301,0.0313,0.0367,0.0086,5.7836,0.0099
4,NO2,9,0.1760,0.0197,0.1592,0.0287,0.0401,6.5985,0.0469
5,NO,9,0.1760,0.0197,0.1592,0.0287,0.0401,6.5985,0.0469
10,temp_sens,2,0.0391,0.0347,0.0131,0.0179,0.1995,inf,0.5046
11,hume_med,2,0.0391,0.0347,0.0131,0.0179,0.1995,inf,0.5046


## 7. Temporal structure of missing values

The temporal distribution of missing observations is examined before selecting an imputation strategy.

For each environmental `LAG0` variable, consecutive sequences of missing observations are identified. Short and isolated gaps are more suitable for linear interpolation than prolonged periods without measurements.

In [11]:
def get_missing_runs(
    dataframe: pd.DataFrame,
    column: str,
    date_column: str = DATE_COLUMN,
) -> pd.DataFrame:
    """Identify consecutive runs of missing observations."""

    data = (
        dataframe[[date_column, column]]
        .copy()
        .sort_values(date_column)
        .reset_index(drop=True)
    )

    data[date_column] = pd.to_datetime(
        data[date_column]
    )

    is_missing = data[column].isna()

    new_run = (
        is_missing
        & (
            ~is_missing.shift(fill_value=False)
            | data[date_column]
            .diff()
            .ne(pd.Timedelta(days=1))
        )
    )

    run_id = new_run.cumsum()

    runs = (
        data.loc[is_missing]
        .groupby(run_id[is_missing])
        .agg(
            start=(date_column, "min"),
            end=(date_column, "max"),
            length=(date_column, "size"),
        )
        .reset_index(drop=True)
        .sort_values(
            ["length", "start"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    return runs

In [12]:
run_summary = []

for col in lag0_cols:
    if not df[col].isna().any():
        continue

    runs = get_missing_runs(
        df,
        column=col,
    )

    longest_run = runs.iloc[0]

    run_summary.append(
        {
            "Variable": col.replace("_LAG0", ""),
            "N_missing": int(df[col].isna().sum()),
            "N_runs": len(runs),
            "Mean_run_length": runs["length"].mean(),
            "Median_run_length": runs["length"].median(),
            "Maximum_run_length": int(longest_run["length"]),
            "Longest_run_start": longest_run["start"].date(),
            "Longest_run_end": longest_run["end"].date(),
        }
    )

run_summary = (
    pd.DataFrame(run_summary)
    .sort_values(
        ["Maximum_run_length", "N_missing"],
        ascending=False,
    )
    .reset_index(drop=True)
)

run_summary.round(2)

,Variable,N_missing,N_runs,Mean_run_length,Median_run_length,Maximum_run_length,Longest_run_start,Longest_run_end
0,CO,136,57,2.39,2.0,16,2017-11-30,2017-12-15
1,O3,16,11,1.45,1.0,3,2014-11-24,2014-11-26
2,PM2p5,48,40,1.20,1.0,2,2019-11-25,2019-11-26
3,PM10,15,13,1.15,1.0,2,2019-11-25,2019-11-26
4,NO,9,7,1.29,1.0,2,2019-11-25,2019-11-26
5,NO2,9,7,1.29,1.0,2,2019-11-25,2019-11-26
6,hume_max,5,5,1.00,1.0,1,2010-03-08,2010-03-08
7,pres_min,3,3,1.00,1.0,1,2013-05-23,2013-05-23
8,pres_med,3,3,1.00,1.0,1,2013-05-23,2013-05-23
9,pres_max,3,3,1.00,1.0,1,2013-05-23,2013-05-23


## 8. Boundary removal and missing-value imputation

The first seven observations are removed because they are affected by the structural boundary introduced by the maximum temporal lag (`LAG-7`).

After removing this boundary, remaining missing values in numerical lagged environmental variables are imputed independently using linear interpolation.

The exploratory analysis above showed that missingness was generally limited and predominantly occurred in short temporal gaps, supporting the interpolation strategy used in the original study.

In [13]:
df_clean = remove_lag_boundary(df)

print(f"Rows before boundary removal: {len(df):,}")
print(f"Rows after boundary removal:  {len(df_clean):,}")
print(f"Rows removed:                {len(df) - len(df_clean)}")

Rows before boundary removal: 5,113
Rows after boundary removal:  5,106
Rows removed:                7


In [14]:
lagged_numeric_cols = [
    col
    for col in df_clean.columns
    if "_LAG" in col
    and pd.api.types.is_numeric_dtype(df_clean[col])
    and df_clean[col].isna().any()
]

n_missing_before = (
    df_clean[lagged_numeric_cols]
    .isna()
    .sum()
    .sum()
)

print(f"Variables requiring imputation: {len(lagged_numeric_cols)}")
print(f"Missing values before imputation: {n_missing_before:,}")

Variables requiring imputation: 96
Missing values before imputation: 1,966


In [15]:
df_clean = impute_lagged_features(df_clean)

n_missing_after = (
    df_clean[lagged_numeric_cols]
    .isna()
    .sum()
    .sum()
)

print(f"Missing values after imputation: {n_missing_after:,}")

Missing values after imputation: 0


## 9. Final data-quality check

After boundary removal and interpolation, the cleaned dataset is checked for any remaining missing observations before being used in subsequent analyses.

In [16]:
remaining_missing = check_remaining_missing_values(
    df_clean
)

if remaining_missing.empty:
    print("✓ No missing values remain in the cleaned dataset.")
else:
    print("Columns with remaining missing values:")
    display(remaining_missing)

✓ No missing values remain in the cleaned dataset.


In [17]:
print(
    f"Final dataset shape: "
    f"{df_clean.shape[0]:,} rows × "
    f"{df_clean.shape[1]} columns"
)

print(
    f"Final date range: "
    f"{df_clean[DATE_COLUMN].min().date()} → "
    f"{df_clean[DATE_COLUMN].max().date()}"
)

Final dataset shape: 5,106 rows × 187 columns
Final date range: 2010-01-08 → 2023-12-31


## 10. Save processed data

The cleaned dataset is saved locally for use in the subsequent exploratory analysis and modelling stages.

Processed data are excluded from version control and are not distributed with this repository.

In [19]:
df_clean.to_excel(
    PROCESSED_DATA_FILE,
    index=False,
)

print(
    f"✓ Cleaned dataset saved to: "
    f"{PROCESSED_DATA_FILE}"
)

✓ Cleaned dataset saved to: C:\Users\migue\Desktop\Healthcare-AI\copd-exacerbation-forecasting\data\processed\copd_daily_data_cleaned.xlsx


### Methodological note

This notebook reproduces the preprocessing strategy used in the original Bachelor Thesis.

Linear interpolation uses neighbouring observations from the time series. For strict prospective forecasting, preprocessing should be performed within each temporal training split to prevent the use of information unavailable at prediction time. This consideration is addressed separately in the modelling pipeline.